In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the cleaned dataset
df = pd.read_csv("football_data_clean.csv")

# Ensure data is sorted chronologically
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

# Encode categorical features (team names & leagues)
label_encoders = {}
for col in ["home_team_name", "away_team_name.x", "league"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le  # Save encoders for later use

# Create lag features (team-specific rolling averages)
def create_lag_features(df, team_column, stat_column, lag=5):
    """Create rolling averages for each team's stat over past matches."""
    df = df.sort_values(["date"])  # Ensure sorted order
    df[f"{stat_column}_lag{lag}"] = df.groupby(team_column)[stat_column].transform(lambda x: x.shift(1).rolling(lag, min_periods=1).mean())
    return df

# Apply lag feature creation for important stats
lag_features = ["PPG_home_last_5", "PPG_away_last_5", "xG_home_team", "xG_away_team", "conversion_rate_home", "conversion_rate_away"]
for stat in lag_features:
    df = create_lag_features(df, "home_team_name", stat)
    df = create_lag_features(df, "away_team_name.x", stat)

# Drop rows where lagged data is missing (first few matches per team)
df = df.dropna()

# Define features (X) and target variable (y)
features = [
    "home_team_name", "away_team_name.x", "league", "home_away_flag",
    "PPG_home_last_5_lag5", "PPG_away_last_5_lag5",  # Lagged values
    "xG_home_team_lag5", "xG_away_team_lag5",
    "conversion_rate_home_lag5", "conversion_rate_away_lag5",
    "average_goals_per_match_pre_match", "average_corners_per_match_pre_match",
    "average_cards_per_match_pre_match", "home_team_shots", "away_team_shots",
    "home_team_shots_on_target", "away_team_shots_on_target",
    "home_team_fouls", "away_team_fouls", "home_team_possession", "away_team_possession",
    "home_team_yellow_cards", "away_team_yellow_cards",
    "home_team_red_cards", "away_team_red_cards", "Corners_home_team", "Corners_away_team",
    "Passes_home_team", "Passes_away_team", "accurate_passes_home_team", "Passes.accurate_away_team",
    "PPDA_home_team", "PPDA_away_team",
    "Match.tempo_home_team", "Match.tempo_away_team",
    "Average.pass.length_home_team", "Average.pass.length_away_team",
    "Shots.outside.PA_home_team", "Shots.outside.PA.on_target_home_team",
    "Positional.attacks_home_team", "Counterattacks_home_team",
    "Defensive.duels_won_home_team", "Defensive.duels.won_away_team"
]

X = df[features]
y = df["result"]  # 1 = Home Win, X = Draw, 2 = Away Win

# Encode target variable into numeric values (1 → 0, X → 1, 2 → 2)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Scale numerical features
scaler = StandardScaler()
X[features] = scaler.fit_transform(X[features])

# Split into train (70%), validation (15%), test (15%) sets
train_size = int(len(df) * 0.7)
val_size = int(len(df) * 0.85)

X_train, y_train = X.iloc[:train_size], y[:train_size]
X_val, y_val = X.iloc[train_size:val_size], y[train_size:val_size]
X_test, y_test = X.iloc[val_size:], y[val_size:]

## **🚀 Train & Tune XGBoost Model**
# Define XGBoost model
xgb_model = XGBClassifier(objective="multi:softmax", num_class=3, eval_metric="mlogloss")

# Define parameter grid for tuning
param_grid = {
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 5, 7],
    "n_estimators": [50, 100, 200]
}

# Perform Grid Search with Cross-Validation
grid_search = GridSearchCV(xgb_model, param_grid, cv=3, scoring="accuracy", verbose=1, n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get best model
best_xgb_model = grid_search.best_estimator_

# Predict on validation set
y_val_pred = best_xgb_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation Accuracy: {val_accuracy:.2f}")
print("\nClassification Report:\n", classification_report(y_val, y_val_pred))


Fitting 3 folds for each of 27 candidates, totalling 81 fits


/var/folders/rg/5qwcvj1934v3254xfppzsxm80000gn/T/ipykernel_466/2255491755.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[features] = scaler.fit_transform(X[features])


Validation Accuracy: 0.60

Classification Report:
               precision    recall  f1-score   support

           0       0.61      0.86      0.71      1386
           1       0.65      0.60      0.62      1060
           2       0.44      0.20      0.27       925

    accuracy                           0.60      3371
   macro avg       0.57      0.55      0.54      3371
weighted avg       0.57      0.60      0.56      3371



In [7]:
# Predict the next match (time series approach)
next_match = X_test.iloc[0:1]  # Predict the first upcoming match
predicted_outcome = best_xgb_model.predict(next_match)

# Convert prediction back to match outcome (1, X, 2)
result_mapping = {0: "1", 1: "X", 2: "2"}
predicted_outcome_label = result_mapping[int(predicted_outcome[0])]

print(f"\nPredicted Outcome for Next Match: {predicted_outcome_label}")



Predicted Outcome for Next Match: 1


/var/folders/rg/5qwcvj1934v3254xfppzsxm80000gn/T/ipykernel_466/3138284678.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[features] = scaler.fit_transform(X[features])


KeyboardInterrupt: 

In [17]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

# Load the dataset
df = pd.read_csv("football_data_clean.csv")

# Ensure chronological order
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

# Filter dataset for Premier League only
df_prem = df[df["league"] == "Premier League"].copy()

# Encode categorical variables
label_encoders = {}
for col in ["home_team_name", "away_team_name.x", "league"]:
    le = LabelEncoder()
    df_prem[col] = le.fit_transform(df_prem[col])
    label_encoders[col] = le

# Define features
features = [
    "home_team_name", "away_team_name.x", "home_away_flag",
    "PPG_home_last_5", "PPG_away_last_5", "PPG_last_5",
    "average_goals_per_match_pre_match", "average_corners_per_match_pre_match",
    "average_cards_per_match_pre_match", "home_team_shots", "away_team_shots",
    "home_team_shots_on_target", "away_team_shots_on_target",
    "home_team_fouls", "away_team_fouls", "home_team_possession", "away_team_possession",
    "home_team_yellow_cards", "away_team_yellow_cards",
    "home_team_red_cards", "away_team_red_cards", "Corners_home_team", "Corners_away_team",
    "Passes_home_team", "Passes_away_team", "accurate_passes_home_team", "Passes.accurate_away_team",
    "xG_home_team", "xG_away_team", "PPDA_home_team", "PPDA_away_team",
    "Match.tempo_home_team", "Match.tempo_away_team",
    "Average.pass.length_home_team", "Average.pass.length_away_team",
    "Shots.outside.PA_home_team", "Shots.outside.PA.on_target_home_team",
    "Positional.attacks_home_team", "Counterattacks_home_team",
    "Defensive.duels_won_home_team", "Defensive.duels.won_away_team",
    "conversion_rate_home", "conversion_rate_away"
]

X = df_prem[features]
y = df_prem["result"]

# Encode target variable ("1", "X", "2") to numerical values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Scale features
scaler = StandardScaler()
X[features] = scaler.fit_transform(X[features])

# Sliding Window Training & Prediction
initial_train_size = int(len(X) * 0.3)
window_size = 100

predictions = []
actual_results = []

# Initial Training
model = XGBClassifier(objective="multi:softmax", num_class=3, eval_metric="mlogloss")

# Progress tracking
from tqdm import tqdm

for i in tqdm(range(initial_train_size, len(X) - 1), desc="Predicting matches"):
    X_train, y_train = X.iloc[max(0, i - window_size):i], y[max(0, i - window_size):i]
    X_next, y_next = X.iloc[i:i+1], y[i]

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_next)

    # Store predictions
    predictions.append(y_pred[0])
    actual_results.append(y_next)

# Convert predictions back to original labels
predicted_results = label_encoder.inverse_transform(predictions)
actual_results = label_encoder.inverse_transform(actual_results)

# Evaluate Performance
accuracy = accuracy_score(actual_results, predicted_results)
print(f"\nModel accuracy on Premier League data: {accuracy:.2f}")
print(classification_report(actual_results, predicted_results))

# Show recent predictions
for i in range(-10, 0):
    print(f"Predicted: {predicted_results[i]}, Actual: {actual_results[i]}")

/var/folders/rg/5qwcvj1934v3254xfppzsxm80000gn/T/ipykernel_466/1369484005.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[features] = scaler.fit_transform(X[features])
Predicting matches: 100%|██████████| 1939/1939 [07:44<00:00,  4.18it/s]


Model accuracy on Premier League data: 0.76
              precision    recall  f1-score   support

           1       0.82      0.89      0.85       858
           2       0.78      0.84      0.81       636
           X       0.56      0.41      0.47       445

    accuracy                           0.76      1939
   macro avg       0.72      0.71      0.71      1939
weighted avg       0.75      0.76      0.75      1939

Predicted: 1, Actual: 1
Predicted: 1, Actual: 1
Predicted: 1, Actual: X
Predicted: 1, Actual: 1
Predicted: X, Actual: 1
Predicted: 1, Actual: 1
Predicted: 2, Actual: 2
Predicted: 2, Actual: 2
Predicted: 1, Actual: 2
Predicted: X, Actual: X
